# Baseline reproduction: CausalPFN on Hillstrom

Reproduces paper **Table 2**, Hillstrom columns. Two tasks, each the full two-arm data (~42.7k rows):

| Task | Paper column | CausalPFN target |
|---|---|---|
| `hill_womens_vs_none` | Hill(1) | 0.992 |
| `hill_mens_vs_none` | Hill(2) | 0.968 |

The code's Hill(1) is the women's e-mail; the paper text says men's. We report by arm name.

Runtime: under an hour on one GPU.

Linear: **TEA-6**. Code: `baseline/` (authors' pinned source in `third_party/CausalPFN`).

## 0. Setup

Finds the repo (or clones it on Colab/Kaggle) and checks Python.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Works when opened from the repo's notebooks/ folder (local Jupyter, college server)
# and on Colab/Kaggle, where it clones the repo first.
BRANCH = "baseline-reproduction"
here = Path.cwd()
if (here / "baseline").exists():
    REPO = here
elif (here.parent / "baseline").exists():
    REPO = here.parent
else:
    REPO = Path("/content" if os.path.exists("/content") else here) / "causal-uplift-engine"
    if not REPO.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--recurse-submodules",
                "-b",
                BRANCH,
                "https://github.com/yashashree5/causal-uplift-engine.git",
                str(REPO),
            ],
            check=True,
        )

if not (REPO / "third_party" / "CausalPFN" / "benchmarks").exists():
    subprocess.run(["git", "submodule", "update", "--init", "--recursive"], cwd=REPO, check=True)

os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("Repo:", REPO)
print("Python:", sys.version.split()[0])
if not ((3, 10) <= sys.version_info[:2] <= (3, 12)):
    print("WARNING: use Python 3.10 (authors' version); 3.11-3.12 usually work, 3.13 does not.")

In [ ]:
# Run once per environment. On Colab/Kaggle, restart the kernel after this cell, then run from step 1 again.
INSTALL = False  # set to True the first time
if INSTALL:
    get_ipython().run_line_magic("pip", "install -q -r baseline/requirements.txt")

## 1. GPU check

In [ ]:
import torch

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    print("WARNING: no GPU found. CausalPFN will be very slow on CPU.")
else:
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}, {p.total_memory / 1e9:.1f} GB")

## 2. Settings

Predictions for this notebook go to their own folder, `outputs/hillstrom/`, separate from the other dataset.

In [ ]:
DATASET = "hillstrom"
TASKS_HERE = ["hill_womens_vs_none", "hill_mens_vs_none"]
OUTPUT_DIR = REPO / "outputs" / DATASET
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Outputs:", OUTPUT_DIR)

## 3. Load the data (authors' preprocessing, 5 folds, seed 82718)

The first run downloads the data through scikit-uplift.

In [ ]:
from baseline.tasks import EXPECTED_ROWS, build

datasets = {}
for task in TASKS_HERE:
    ds = build(task)
    datasets[task] = ds
    n = len(ds.all_X)
    off = abs(n - EXPECTED_ROWS[task]) / EXPECTED_ROWS[task] > 0.05
    flag = "  <-- differs from expected, note in TEA-6" if off else ""
    print(f"{task}: rows={n:,} (expected ~{EXPECTED_ROWS[task]:,}){flag}")
    print(f"   features={ds.all_X.shape[1]}  treated={ds.all_t.mean():.3f}", end="  ")
    print(f"outcome rate={ds.all_y.mean():.4f}")
    print(f"   fold test sizes={[len(i) for i in ds.all_fold_test_idx]}")

## 4. Run CausalPFN and the four comparison methods

Each fold is saved as it finishes. If the kernel stops, re-run steps 0–2 and this cell: finished folds are skipped.

If you see **`CUDA out of memory`**, note the GPU in TEA-6 and use a larger GPU. Don't lower the context size.

In [ ]:
import time

from baseline.run import METHODS, _set_seeds, run_task

TASKS_TO_RUN = TASKS_HERE

_set_seeds()
start = time.time()
for task in TASKS_TO_RUN:
    run_task(task, METHODS, OUTPUT_DIR, DEVICE)
print(f"\nTotal time: {(time.time() - start) / 60:.1f} min")

## 5. Results vs. the paper

Normalized Qini: each method's score divided by the best method's score for that task. Pass: within ±0.03 of the paper.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

from baseline.evaluate import score_task, to_markdown

tables = [score_task(OUTPUT_DIR, t) for t in TASKS_HERE]
tables = [t for t in tables if not t.empty]
if not tables:
    print("No finished tasks yet (all 5 folds needed per method).")
else:
    table = pd.concat(tables, ignore_index=True)
    report_md = to_markdown(DATASET, table)
    display(Markdown(report_md))
    REPORTS = REPO / "reports"
    REPORTS.mkdir(exist_ok=True)
    table.to_csv(REPORTS / f"baseline_{DATASET}.csv", index=False)
    (REPORTS / f"baseline_{DATASET}.md").write_text(report_md)
    print("Saved:", REPORTS / f"baseline_{DATASET}.md")

## 6. Qini curves

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from baseline.evaluate import load
from eval.qini import get_qini_curve

done = [t for t in TASKS_HERE if load(OUTPUT_DIR, t)]
fig, axes = plt.subplots(1, max(len(done), 1), figsize=(5.5 * max(len(done), 1), 4), squeeze=False)
for ax, task in zip(axes[0], done, strict=False):
    for method, df in load(OUTPUT_DIR, task).items():
        curve, score = get_qini_curve(
            df["t"].to_numpy(float), df["y"].to_numpy(float), df["uplift_pred"].to_numpy(float)
        )
        x = np.linspace(0, 1, len(curve))
        ax.plot(x, curve, label=f"{method} ({score:.3f})", lw=2 if method == "causalpfn" else 1)
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="random")
    ax.set_title(task)
    ax.set_xlabel("Fraction of customers targeted")
    ax.set_ylabel("Normalized Qini")
    ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(REPO / "reports" / f"qini_{DATASET}.png", dpi=150)
plt.show()

## 7. After the run

1. Commit `reports/baseline_hillstrom.md`, `reports/baseline_hillstrom.csv`, and `reports/qini_hillstrom.png` to the `baseline-reproduction` branch with `Part of TEA-6` in the message.
2. Comment on **TEA-6** with the GPU name, total runtime, any row-count warning, and the CausalPFN numbers.